# Generate a 12-security demo workbook with yfinance

Run the code cell below in Colab or Jupyter. It creates a polished Excel demo workbook with:
- 12 securities
- 2 block sizes per security
- live 12-month market data from yfinance
- polished demo-facing sheets
- technical covariance and price history sheets

In [ ]:

# Run this whole cell in Colab or Jupyter.
# It downloads 12 months of market data with yfinance and builds
# a polished demo workbook for 12 securities x 2 block sizes each.

!pip -q install yfinance openpyxl pandas numpy

from pathlib import Path
from math import sqrt
import numpy as np
import pandas as pd
import yfinance as yf
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.comments import Comment
from openpyxl.worksheet.table import Table, TableStyleInfo

# -------------------------
# Configuration
# -------------------------
TICKERS = [
    "NVDA", "AAPL", "MSFT", "AVGO", "MU", "ORCL",
    "AMD", "LRCX", "INTC", "CSCO", "AMAT", "KLAC"
]
BLOCK_TARGETS = [100000, 150000]
BUDGET_USD = 1_000_000
RISK_FREE_RATE = 0.04
LAMBDA_BUDGET = 12.0
LAMBDA_VARIANCE = 6.0
LAMBDA_EXCLUSIVE = 20.0
OUTPUT_PATH = Path("demo_budget_block_options_12sec.xlsx")

NAME_MAP = {
    "NVDA": "NVIDIA",
    "AAPL": "Apple",
    "MSFT": "Microsoft",
    "AVGO": "Broadcom",
    "MU": "Micron Technology",
    "ORCL": "Oracle",
    "AMD": "Advanced Micro Devices",
    "LRCX": "Lam Research",
    "INTC": "Intel",
    "CSCO": "Cisco",
    "AMAT": "Applied Materials",
    "KLAC": "KLA",
}

# -------------------------
# Download data
# -------------------------
prices = yf.download(
    tickers=TICKERS,
    period="12mo",
    interval="1d",
    auto_adjust=True,
    progress=False,
)["Close"]

if isinstance(prices, pd.Series):
    prices = prices.to_frame()

prices = prices.dropna(how="all").ffill().dropna()
rets = prices.pct_change().dropna()

total_return_12m = prices.iloc[-1] / prices.iloc[0] - 1
ann_vol = rets.std() * np.sqrt(252)
mean_daily = rets.mean()
std_daily = rets.std()
daily_cov = rets.cov()
annual_cov = daily_cov * 252
latest_price = prices.iloc[-1]

# -------------------------
# Build workbook
# -------------------------
wb = Workbook()
dark = PatternFill("solid", fgColor="1F4E78")
section = PatternFill("solid", fgColor="D9EAF7")
input_fill = PatternFill("solid", fgColor="FFF2CC")
white_bold = Font(color="FFFFFF", bold=True)
bold = Font(bold=True)
blue = Font(color="0000FF")
green = Font(color="008000")
gray = Font(color="666666")
wrap = Alignment(wrap_text=True, vertical="top")

# ReadMe
ws = wb.active
ws.title = "ReadMe"
ws["A1"] = "12-security budget-first QAOA demo workbook"
ws["A1"].fill = dark
ws["A1"].font = white_bold
ws.merge_cells("A1:H1")
ws["A2"] = (
    "This workbook is a polished demo input for a budget-first allocation optimizer. "
    "It contains 12 securities, 2 allowed block sizes per security, and live market data pulled with yfinance "
    "for the last 12 months. The result sheets are included as placeholders for a solver notebook."
)
ws["A2"].alignment = wrap
ws.merge_cells("A2:H3")
lines = [
    "Universe: 12 large-cap S&P 500 Information Technology names.",
    "Decision design: 2 allowed block sizes per security: USD 100k and USD 150k.",
    "Primary business constraint: spend about the configured budget, not a fixed asset count.",
    "Market data source: yfinance (12-month adjusted close history).",
    "Technical result sheets are placeholders in this demo workbook and can be filled by a solver notebook.",
]
for i, line in enumerate(lines, start=5):
    ws[f"A{i}"] = line
for c in "ABCDEFGH":
    ws.column_dimensions[c].width = 22

# Assets
assets = wb.create_sheet("Assets")
assets["A1"] = "Asset universe"
assets["A1"].fill = dark
assets["A1"].font = white_bold
assets.merge_cells("A1:J1")
asset_headers = [
    "Ticker", "Company", "Current Price (USD)", "Price Source Status",
    "12M Return Proxy", "Annual Volatility", "Mean Daily Return",
    "Std Daily Return", "Allowed", "Source URL"
]
for j, h in enumerate(asset_headers, start=1):
    c = assets.cell(2, j, h)
    c.fill = dark
    c.font = white_bold
    c.alignment = wrap

for i, ticker in enumerate(TICKERS, start=3):
    assets.cell(i, 1, ticker)
    assets.cell(i, 2, NAME_MAP.get(ticker, ticker))
    assets.cell(i, 3, float(latest_price[ticker]))
    assets.cell(i, 4, "Imported via yfinance")
    assets.cell(i, 5, float(total_return_12m[ticker]))
    assets.cell(i, 6, float(ann_vol[ticker]))
    assets.cell(i, 7, float(mean_daily[ticker]))
    assets.cell(i, 8, float(std_daily[ticker]))
    assets.cell(i, 9, 1)
    assets.cell(i, 10, "https://pypi.org/project/yfinance/")
    assets.cell(i, 3).number_format = '$#,##0.00_);[Red]($#,##0.00)'
    assets.cell(i, 5).number_format = '0.0%'
    assets.cell(i, 6).number_format = '0.0%'
    assets.cell(i, 7).number_format = '0.0000'
    assets.cell(i, 8).number_format = '0.0000'
    assets.cell(i, 4).font = green
    assets.cell(i, 9).font = blue
    assets.cell(i, 9).fill = input_fill

tab = Table(displayName="AssetsTable", ref=f"A2:J{2+len(TICKERS)}")
tab.tableStyleInfo = TableStyleInfo(name="TableStyleMedium2", showFirstColumn=False, showLastColumn=False, showRowStripes=True, showColumnStripes=False)
assets.add_table(tab)
for col, width in {"A":12, "B":24, "C":18, "D":18, "E":16, "F":16, "G":16, "H":16, "I":10, "J":30}.items():
    assets.column_dimensions[col].width = width
assets.freeze_panes = "A3"

# Settings
settings = wb.create_sheet("Settings")
settings["A1"] = "Model settings"
settings["A1"].fill = dark
settings["A1"].font = white_bold
settings.merge_cells("A1:C1")
for c, v in [("A2", "Key"), ("B2", "Value"), ("C2", "Description")]:
    settings[c] = v
    settings[c].fill = section
    settings[c].font = bold
settings_rows = [
    ("budget_usd", BUDGET_USD, "Target total budget"),
    ("risk_free_rate_annual", RISK_FREE_RATE, "Annual risk-free rate used in excess-return reward"),
    ("lambda_budget", LAMBDA_BUDGET, "Budget deviation penalty"),
    ("lambda_variance", LAMBDA_VARIANCE, "Variance contribution weight"),
    ("lambda_exclusive", LAMBDA_EXCLUSIVE, "Penalty for selecting more than one block option of same asset"),
    ("top_n_export", 20, "Target number of candidate portfolios to export"),
]
for i, (k, v, d) in enumerate(settings_rows, start=3):
    settings.cell(i, 1, k)
    settings.cell(i, 2, v)
    settings.cell(i, 3, d)
    settings.cell(i, 2).font = blue
    settings.cell(i, 2).fill = input_fill
settings["B3"].number_format = '$#,##0'
settings["B4"].number_format = '0.0%'
for r in [5, 6, 7]:
    settings[f"B{r}"].number_format = '0.00'
for col in ["A", "B", "C"]:
    settings.column_dimensions[col].width = 24

# BlockOptions with literal values, no formulas
blocks = wb.create_sheet("BlockOptions")
blocks["A1"] = "Allowed block-size choices"
blocks["A1"].fill = dark
blocks["A1"].font = white_bold
blocks.merge_cells("A1:J1")
block_headers = [
    "decision_id", "Ticker", "Block Label", "Target Block USD",
    "Shares", "Approx Cost USD", "Allowed", "Asset Group",
    "Expected Return Proxy", "Annual Volatility"
]
for j, h in enumerate(block_headers, start=1):
    c = blocks.cell(2, j, h)
    c.fill = dark
    c.font = white_bold
    c.alignment = wrap

r = 3
for ticker in TICKERS:
    price = float(latest_price[ticker])
    for target in BLOCK_TARGETS:
        shares = max(1, int(round(target / price)))
        approx_cost = shares * price
        label = f"{int(target/1000)}k"
        blocks.cell(r, 1, f"{ticker}_{label}")
        blocks.cell(r, 2, ticker)
        blocks.cell(r, 3, label)
        blocks.cell(r, 4, float(target))
        blocks.cell(r, 5, int(shares))
        blocks.cell(r, 6, float(approx_cost))
        blocks.cell(r, 7, 1)
        blocks.cell(r, 8, ticker)
        blocks.cell(r, 9, float(total_return_12m[ticker]))
        blocks.cell(r, 10, float(ann_vol[ticker]))
        blocks.cell(r, 4).number_format = '$#,##0'
        blocks.cell(r, 6).number_format = '$#,##0.00_);[Red]($#,##0.00)'
        blocks.cell(r, 9).number_format = '0.0%'
        blocks.cell(r, 10).number_format = '0.0%'
        blocks.cell(r, 7).font = blue
        blocks.cell(r, 7).fill = input_fill
        r += 1

tab = Table(displayName="BlockOptionsTable", ref=f"A2:J{r-1}")
tab.tableStyleInfo = TableStyleInfo(name="TableStyleMedium2", showFirstColumn=False, showLastColumn=False, showRowStripes=True, showColumnStripes=False)
blocks.add_table(tab)
for col, width in {"A":18, "B":10, "C":12, "D":16, "E":10, "F":16, "G":10, "H":12, "I":18, "J":16}.items():
    blocks.column_dimensions[col].width = width
blocks.freeze_panes = "A3"

# Returns
returns_ws = wb.create_sheet("Returns")
returns_ws["A1"] = "12M statistics"
returns_ws["A1"].fill = dark
returns_ws["A1"].font = white_bold
returns_ws.merge_cells("A1:G1")
ret_headers = ["Ticker", "12M Total Return", "Annualized Volatility", "Mean Daily Return", "Std Daily Return", "Status", "Source URL"]
for j, h in enumerate(ret_headers, start=1):
    returns_ws.cell(2, j, h).fill = section
    returns_ws.cell(2, j).font = bold
for i, ticker in enumerate(TICKERS, start=3):
    returns_ws.cell(i, 1, ticker)
    returns_ws.cell(i, 2, float(total_return_12m[ticker]))
    returns_ws.cell(i, 3, float(ann_vol[ticker]))
    returns_ws.cell(i, 4, float(mean_daily[ticker]))
    returns_ws.cell(i, 5, float(std_daily[ticker]))
    returns_ws.cell(i, 6, "Imported via yfinance")
    returns_ws.cell(i, 7, "https://pypi.org/project/yfinance/")
    returns_ws.cell(i, 2).number_format = '0.0%'
    returns_ws.cell(i, 3).number_format = '0.0%'
for col in "ABCDEFG":
    returns_ws.column_dimensions[col].width = 20

# Covariance sheets
for sheet_name, matrix, title in [
    ("Covariance", daily_cov, "Daily return covariance matrix"),
    ("AnnualizedCovariance", annual_cov, "Annualized covariance matrix"),
]:
    sh = wb.create_sheet(sheet_name)
    sh["A1"] = title
    sh["A1"].fill = dark
    sh["A1"].font = white_bold
    sh.merge_cells(f"A1:{chr(65+len(TICKERS))}1")
    for j, ticker in enumerate(TICKERS, start=2):
        sh.cell(2, j, ticker)
        sh.cell(2, j).fill = section
        sh.cell(2, j).font = bold
    for i, tr in enumerate(TICKERS, start=3):
        sh.cell(i, 1, tr)
        sh.cell(i, 1).fill = section
        sh.cell(i, 1).font = bold
        for j, tc in enumerate(TICKERS, start=2):
            sh.cell(i, j, float(matrix.loc[tr, tc]))
            sh.cell(i, j).number_format = "0.0000"
    sh.freeze_panes = "B3"
    for col_idx in range(1, len(TICKERS)+2):
        sh.column_dimensions[chr(64+col_idx)].width = 16

# PriceHistory
ph = wb.create_sheet("PriceHistory")
ph["A1"] = "Adjusted close price history"
ph["A1"].fill = dark
ph["A1"].font = white_bold
ph.merge_cells(f"A1:{chr(65+len(TICKERS))}1")
ph["A2"] = "Date"
ph["A2"].fill = section
ph["A2"].font = bold
for j, ticker in enumerate(TICKERS, start=2):
    ph.cell(2, j, ticker)
    ph.cell(2, j).fill = section
    ph.cell(2, j).font = bold
for i, dt in enumerate(prices.index, start=3):
    ph.cell(i, 1, dt.to_pydatetime())
    for j, ticker in enumerate(TICKERS, start=2):
        ph.cell(i, j, float(prices.loc[dt, ticker]))
for col_idx in range(1, len(TICKERS)+2):
    ph.column_dimensions[chr(64+col_idx)].width = 16
ph.freeze_panes = "B3"

# Demo-facing placeholder result sheets
for name, title in [
    ("Results_Summary", "Results summary"),
    ("Results_Overview", "Candidate portfolios overview"),
    ("Results_Portfolios", "Exploded portfolio details"),
    ("Solver_Comparison", "Solver comparison"),
]:
    sh = wb.create_sheet(name)
    sh["A1"] = title
    sh["A1"].fill = dark
    sh["A1"].font = white_bold
    sh["A2"] = "This sheet is a placeholder in the demo workbook and can be filled by the solver notebook."
    sh["A2"].font = gray

# Sources
src = wb.create_sheet("Sources")
src["A1"] = "Sources"
src["A1"].fill = dark
src["A1"].font = white_bold
src["A2"] = "Historical market data"
src["B2"] = "https://pypi.org/project/yfinance/"
src["A3"] = "Universe style reference"
src["B3"] = "https://www.slickcharts.com/sp500"
src["A4"] = "Generated"
src["B4"] = "This workbook was generated by a Python script using yfinance."

# Comments
comment_text = (
    "Imported using yfinance. Values depend on the runtime date and the last 12 months of adjusted-close history."
)
assets["C3"].comment = Comment(comment_text, "OpenAI")
blocks["F3"].comment = Comment("Literal value generated from shares x latest imported price.", "OpenAI")

wb.save(OUTPUT_PATH)
print(f"Workbook created: {OUTPUT_PATH.resolve()}")
